# 04 Dialect Adaptation

Dialect/adaptation analysis for Common Voice Yue. This notebook does not train adapters yet; it identifies which metadata groups and error categories should drive adapter, LoRA, accent-embedding, and soft-label experiments.


## Setup


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

def find_project_root():
    cwd = Path.cwd().resolve()
    env_root = os.getenv("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path(r"D:/GitHub/Cantonese-Speech-AI"),
        Path(r"D:\GitHub\Cantonese-Speech-AI"),
        Path("/mnt/d/GitHub/Cantonese-Speech-AI"),
        Path("/content/Cantonese-Speech-AI"),
        Path("/content/drive/MyDrive/Cantonese-Speech-AI"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "cantonese_speech_ai").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot find project root from cwd={cwd}")

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
load_dotenv(ROOT / ".env")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

ROOT


WindowsPath('D:/GitHub/Cantonese-Speech-AI')

## Load baseline predictions and tone taxonomy


In [2]:
prediction_path = ROOT / "outputs" / "predictions" / "whisper_api_dev_20.csv"
taxonomy_path = ROOT / "outputs" / "analysis" / "tone_error_taxonomy_dev_20.csv"

if not prediction_path.exists():
    prediction_files = sorted(
        (ROOT / "outputs" / "predictions").glob("whisper_api_dev_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError("Run 02_asr_baseline.ipynb before this notebook.")
    prediction_path = prediction_files[0]

predictions = pd.read_csv(prediction_path, encoding="utf-8-sig")
taxonomy = pd.read_csv(taxonomy_path, encoding="utf-8-sig") if taxonomy_path.exists() else pd.DataFrame()

analysis = predictions.copy()
if not taxonomy.empty:
    taxonomy_cols = [
        "path",
        "error_category",
        "written_replacement_flags",
        "reference_jyutping",
        "prediction_jyutping",
        "reference_tones",
        "prediction_tones",
        "tone_count_delta",
    ]
    analysis = analysis.merge(taxonomy[taxonomy_cols], on="path", how="left")

analysis["sentence_length"] = analysis["sentence"].fillna("").str.len()
analysis["accents"] = analysis["accents"].fillna("unknown").replace("", "unknown")
analysis["age"] = analysis["age"].fillna("unknown").replace("", "unknown")
analysis["gender"] = analysis["gender"].fillna("unknown").replace("", "unknown")
analysis["error_category"] = analysis["error_category"].fillna("unclassified")
analysis["written_replacement_flags"] = analysis["written_replacement_flags"].fillna("[]")
analysis["has_written_replacement"] = analysis["written_replacement_flags"].ne("[]")
analysis["tone_count_delta"] = analysis["tone_count_delta"].fillna(0)
analysis["high_cer"] = analysis["cer"] >= 0.25

{
    "prediction_path": str(prediction_path),
    "taxonomy_path": str(taxonomy_path),
    "rows": len(analysis),
    "mean_cer": analysis["cer"].mean(),
    "mean_wer": analysis["wer"].mean(),
}


{'prediction_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\predictions\\whisper_api_dev_20.csv',
 'taxonomy_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\tone_error_taxonomy_dev_20.csv',
 'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'mean_wer': np.float64(0.8833333333333334)}

## Group risk analysis


In [3]:
def group_profile(frame, column):
    profile = (
        frame.groupby(column, dropna=False)
        .agg(
            count=("path", "count"),
            mean_cer=("cer", "mean"),
            mean_wer=("wer", "mean"),
            high_cer_rate=("high_cer", "mean"),
            written_style_rate=("has_written_replacement", "mean"),
            mean_abs_tone_delta=("tone_count_delta", lambda s: s.abs().mean()),
        )
        .reset_index()
        .rename(columns={column: "group_value"})
    )
    profile.insert(0, "group_type", column)
    return profile.sort_values(["mean_cer", "count"], ascending=[False, False])

group_profiles = pd.concat(
    [
        group_profile(analysis, "accents"),
        group_profile(analysis, "age"),
        group_profile(analysis, "gender"),
        group_profile(analysis, "error_category"),
    ],
    ignore_index=True,
)
group_profiles


,group_type,group_value,count,mean_cer,mean_wer,high_cer_rate,written_style_rate,mean_abs_tone_delta
0,accents,unknown,20,0.306781,0.883333,0.6,0.35,0.300000
1,age,thirties,20,0.306781,0.883333,0.6,0.35,0.300000
2,gender,female_feminine,20,0.306781,0.883333,0.6,0.35,0.300000
3,error_category,high_cer_written_style,6,0.535516,1.000000,1.0,1.00,0.333333
4,error_category,high_cer,6,0.341558,1.166667,1.0,0.00,0.333333
5,error_category,written_style_replacement,1,0.176471,0.500000,0.0,1.00,1.000000
6,error_category,low_error,7,0.099529,0.595238,0.0,0.00,0.142857


## Adaptation recommendations


In [4]:
def recommend(row):
    if row["count"] < 3:
        return "collect_more_samples"
    if row["written_style_rate"] >= 0.4:
        return "soft_label_transfer_with_colloquial_preservation"
    if row["mean_abs_tone_delta"] >= 2:
        return "tone_aware_adapter"
    if row["mean_cer"] >= 0.25:
        return "accent_lora_or_sparse_adapter"
    return "monitor_baseline"

adaptation_plan = group_profiles.copy()
adaptation_plan["recommended_method"] = adaptation_plan.apply(recommend, axis=1)
adaptation_plan["teacher_signal"] = "whisper_api_prediction"
adaptation_plan["student_target"] = "common_voice_yue_transcript"
adaptation_plan["notes"] = (
    "Use as exploratory ranking only; run larger dev sample before training adapters."
)

plan_path = ROOT / "outputs" / "analysis" / "dialect_adaptation_plan_dev_20.csv"
plan_path.parent.mkdir(parents=True, exist_ok=True)
adaptation_plan.to_csv(plan_path, index=False, encoding="utf-8-sig")

{
    "plan_path": str(plan_path),
    "rows": len(adaptation_plan),
    "methods": adaptation_plan["recommended_method"].value_counts().to_dict(),
}


{'plan_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\dialect_adaptation_plan_dev_20.csv',
 'rows': 7,
 'methods': {'accent_lora_or_sparse_adapter': 4,
  'soft_label_transfer_with_colloquial_preservation': 1,
  'collect_more_samples': 1,
  'monitor_baseline': 1}}

In [5]:
adaptation_plan[
    [
        "group_type",
        "group_value",
        "count",
        "mean_cer",
        "high_cer_rate",
        "written_style_rate",
        "mean_abs_tone_delta",
        "recommended_method",
    ]
]


,group_type,group_value,count,mean_cer,high_cer_rate,written_style_rate,mean_abs_tone_delta,recommended_method
0,accents,unknown,20,0.306781,0.6,0.35,0.300000,accent_lora_or_sparse_adapter
1,age,thirties,20,0.306781,0.6,0.35,0.300000,accent_lora_or_sparse_adapter
2,gender,female_feminine,20,0.306781,0.6,0.35,0.300000,accent_lora_or_sparse_adapter
3,error_category,high_cer_written_style,6,0.535516,1.0,1.00,0.333333,soft_label_transfer_with_colloquial_preservation
4,error_category,high_cer,6,0.341558,1.0,0.00,0.333333,accent_lora_or_sparse_adapter
5,error_category,written_style_replacement,1,0.176471,0.0,1.00,1.000000,collect_more_samples
6,error_category,low_error,7,0.099529,0.0,0.00,0.142857,monitor_baseline


## Adapter experiment matrix


In [6]:
experiment_matrix = pd.DataFrame(
    [
        {
            "experiment": "baseline_whisper_api",
            "trainable_params": "none",
            "signal": "zero-shot teacher",
            "evaluation": "CER/WER + written-style rate",
        },
        {
            "experiment": "accent_lora",
            "trainable_params": "LoRA modules",
            "signal": "gold transcript + teacher soft label",
            "evaluation": "group CER by accents/age/gender",
        },
        {
            "experiment": "tone_aware_adapter",
            "trainable_params": "encoder adapter + tone auxiliary head",
            "signal": "Jyutping/tone scaffold from 03",
            "evaluation": "CER + tone_count_delta + colloquial marker retention",
        },
        {
            "experiment": "colloquial_preservation_student",
            "trainable_params": "student decoder/adapters",
            "signal": "teacher transcript cleaned against Common Voice labels",
            "evaluation": "written-style replacement rate",
        },
    ]
)
experiment_matrix


,experiment,trainable_params,signal,evaluation
0,baseline_whisper_api,none,zero-shot teacher,CER/WER + written-style rate
1,accent_lora,LoRA modules,gold transcript + teacher soft label,group CER by accents/age/gender
2,tone_aware_adapter,encoder adapter + tone auxiliary head,Jyutping/tone scaffold from 03,CER + tone_count_delta + colloquial marker ret...
3,colloquial_preservation_student,student decoder/adapters,teacher transcript cleaned against Common Voic...,written-style replacement rate
